# 🏙️ Traffy Fondue Machine Learning & Specialized Sub-models Pipeline
### ระบบ AI พยากรณ์ระยะเวลาซ่อมแซมปัญหาเมือง กทม. และจัดลำดับความสำคัญเชิงรุก (Actionable Decision Layer)

สมุดโค้ด (Notebook) นี้แสดงการทำงานครบวงจรของโปรเจกต์:
1. **Data Inspection:** สำรวจข้อมูลจริงที่คลีนสมบูรณ์แล้วใน `data/processed/` (กว่า 275,000 เคส)
2. **11 Front-End Features:** วิเคราะห์ตัวแปรหน้างานที่สกัดได้ ณ วินาทีแรกที่ประชาชนกดแจ้งเรื่อง
3. **Specialized Sub-models (Mixture of Experts):** ระบบ Router กระจายงานเข้า 6 โมเดลตามความเชี่ยวชาญของแต่ละหน่วยงาน
4. **Head-to-Head Comparison:** วัดผลเปรียบเทียบระหว่างโมเดลเดี่ยว (Single Model) vs โมเดลเฉพาะฝ่าย บน Test Set 49,010 เคส
5. **Actionable Decision Layer:** คำนวณ Public Impact Score และจัดกลุ่มปัญหาลง **BMA Action Triage Matrix (4 Quadrants)**
6. **Interactive Demo:** ฟังก์ชันจำลองการรับเรื่องร้องเรียนจริงแบบ Real-time


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, median_absolute_error

# กำหนด Path หลักของโปรเจกต์
BASE_DIR = os.getcwd()
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
SPECIALIZED_DIR = os.path.join(MODELS_DIR, "specialized")

print("Python Environment Ready!")
print(f"Base Directory: {BASE_DIR}")
print(f"Cleaned Data Directory: {PROCESSED_DIR}")


---
## 1. การโหลดข้อมูลที่คลีนแล้ว (Cleaned Datasets)
ข้อมูลผ่านขั้นตอน Data Cleaning (ตัด Outliers, คำนวณ `duration_days` และสกัด 11 Features หน้างาน) และจัดเก็บในรูปแบบ **Parquet** ที่โหลดได้ภายในเวลาไม่ถึง 1 วินาที


In [ ]:
# โหลดชุดข้อมูลทั้ง 3 ชุดจาก data/processed/
train_path = os.path.join(PROCESSED_DIR, "train_cleaned.parquet")
val_path = os.path.join(PROCESSED_DIR, "val_cleaned.parquet")
test_path = os.path.join(PROCESSED_DIR, "test_cleaned.parquet")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f"📊 1. Train Set (ปี 2023 ทั้งปี)         : {len(train_df):,} แถว")
print(f"📊 2. Validation Set (ต้นปี 2024: ม.ค.-มี.ค.): {len(val_df):,} แถว")
print(f"📊 3. Test Set (กลางปี 2024: เม.ย.-มิ.ย.)   : {len(test_df):,} แถว")
print(f"--> รวมข้อมูลทั้งหมดที่ใช้ในการทดลอง      : {len(train_df) + len(val_df) + len(test_df):,} แถว")

# แสดงตัวอย่างข้อมูล 5 แถวแรก
train_df[['ticket_id', 'district', 'main_type', 'sub_category', 'predicted_dept', 'severity', 'duration_days']].head()


---
## 2. ตัวแปรต้น 11 Features หน้างาน (Feature Matrix: ตัวแปร X)
เป็นข้อมูลที่ระบบสามารถทราบได้ **ณ วินาทีแรกที่ประชาชนกดแจ้งเรื่อง** ไม่ใช้ข้อมูลอนาคต (No Data Leakage)


In [ ]:
feature_cols = [
    'district', 'main_type', 'sub_category', 'predicted_dept',
    'severity', 'comment_len', 'day_of_week', 'is_weekend',
    'month', 'is_rainy_season', 'hour'
]

print("รายชื่อ 11 Features หน้างาน:")
for i, col in enumerate(feature_cols, 1):
    dtype = train_df[col].dtype
    n_unique = train_df[col].nunique()
    print(f"  {i:2d}. {col:<18} | ชนิดข้อมูล: {str(dtype):<10} | จำนวนค่าไม่ซ้ำ: {n_unique}")

# ตรวจสอบสถิติพื้นฐานของระยะเวลาซ่อมจริง (duration_days)
print("
สถิติของ Target (ระยะเวลาซ่อมจริง - วัน):")
print(train_df['duration_days'].describe(percentiles=[0.25, 0.50, 0.75, 0.90]))


---
## 3. สถาปัตยกรรม Specialized Sub-models (Mixture of Experts)
แทนที่จะใช้โมเดลเดี่ยวรวมทุกฝ่าย ระบบได้ฝึกสอนโมเดลเฉพาะทางของ **5 ฝ่ายหลัก** + **1 General Fallback Model**:
* `model_yotha.txt`: ฝ่ายโยธา
* `model_cleanliness.txt`: ฝ่ายรักษาความสะอาดฯ
* `model_thetsakit.txt`: ฝ่ายเทศกิจ
* `model_environment.txt`: ฝ่ายสิ่งแวดล้อมฯ
* `model_drainage.txt`: สำนักการระบายน้ำ
* `model_general.txt`: General Fallback


In [ ]:
from src.specialized_predictor import SpecializedBMAPredictor

# เรียกใช้งาน SpecializedBMAPredictor
predictor = SpecializedBMAPredictor(models_dir=SPECIALIZED_DIR)

print(f"ความพร้อมของระบบ Specialized Models: {predictor.is_ready()}")
print(f"รายชื่อโมเดลย่อยที่โหลดเข้าสู่ RAM : {list(predictor.models.keys())}")
print(f"การแปลงชื่อหน่วยงาน (Slug Map)       : {predictor.dept_map}")


---
## 4. วัดผลเปรียบเทียบ: โมเดลเดี่ยว (Single) vs โมเดลเฉพาะฝ่าย (Specialized)
ทดสอบบนชุดทดสอบจริง **Holdout Test Set ปี 2024 (49,010 เคส)**


In [ ]:
# 1. ทำนายด้วย Single Model ตัวเดิม
single_model_path = os.path.join(MODELS_DIR, "lightgbm_traffy_real.txt")
single_cats_path = os.path.join(MODELS_DIR, "categories.json")

single_model = lgb.Booster(model_file=single_model_path)
with open(single_cats_path, "r", encoding="utf-8") as f:
    single_cats = json.load(f)

test_eval_df = test_df.copy()
for c in ['district', 'main_type', 'sub_category', 'predicted_dept']:
    test_eval_df[c] = pd.Categorical(test_eval_df[c], categories=single_cats.get(c, []))

single_preds_log = single_model.predict(test_eval_df[feature_cols])
test_eval_df['single_pred_days'] = np.expm1(single_preds_log)

# 2. ทำนายด้วย Specialized Sub-models ผ่าน Router
test_eval_df = predictor.predict_df(test_eval_df)

# คำนวณ Metric รวม
actual_days = test_eval_df['duration_days'].values
single_mae = mean_absolute_error(actual_days, test_eval_df['single_pred_days'])
single_med_ae = median_absolute_error(actual_days, test_eval_df['single_pred_days'])

spec_mae = mean_absolute_error(actual_days, test_eval_df['specialized_pred_days'])
spec_med_ae = median_absolute_error(actual_days, test_eval_df['specialized_pred_days'])

single_acc3 = np.mean(np.abs(actual_days - test_eval_df['single_pred_days']) <= 3.0) * 100
spec_acc3 = np.mean(np.abs(actual_days - test_eval_df['specialized_pred_days']) <= 3.0) * 100

print("=" * 70)
print("🏆 สรุปผลการแข่งขันภาพรวม (Overall Benchmark บน 49,010 เคส)")
print("=" * 70)
print(f"1. MAE (คลาดเคลื่อนเฉลี่ย)       : Single = {single_mae:5.2f} วัน | Specialized = {spec_mae:5.2f} วัน")
print(f"2. Median AE (มัธยฐานคลาดเคลื่อน) : Single = {single_med_ae:5.2f} วัน | Specialized = {spec_med_ae:5.2f} วัน")
print(f"3. ความแม่นยำในกรอบ ±3 วัน       : Single = {single_acc3:5.2f}% | Specialized = {spec_acc3:5.2f}%")
print("=" * 70)


---
## 5. ผลการประเมินเจาะลึกรายหน่วยงาน (Per-Department Breakdown)
ตารางเปรียบเทียบผลลัพธ์แยกตามหน่วยงานที่รับผิดชอบ แสดงให้เห็นว่าโมเดลเฉพาะฝ่ายทำงานได้แม่นยำขึ้นอย่างมีนัยสำคัญ


In [ ]:
dept_stats = []
target_depts = ['ฝ่ายเทศกิจ', 'ฝ่ายรักษาความสะอาดฯ', 'สำนักการระบายน้ำ', 'ฝ่ายสิ่งแวดล้อมฯ', 'ฝ่ายโยธา']

for d in target_depts:
    sub = test_eval_df[test_eval_df['predicted_dept'] == d]
    if len(sub) == 0:
        continue
    act = sub['duration_days'].values
    s_mae = mean_absolute_error(act, sub['single_pred_days'])
    s_med = median_absolute_error(act, sub['single_pred_days'])
    p_mae = mean_absolute_error(act, sub['specialized_pred_days'])
    p_med = median_absolute_error(act, sub['specialized_pred_days'])
    improvement = ((s_mae - p_mae) / s_mae) * 100

    dept_stats.append({
        'ฝ่ายงาน': d,
        'จำนวนเคส': len(sub),
        'Single MAE': round(s_mae, 2),
        'Spec MAE': round(p_mae, 2),
        'Single MedAE': round(s_med, 2),
        'Spec MedAE': round(p_med, 2),
        'MAE Improvement (%)': round(improvement, 2)
    })

comparison_df = pd.DataFrame(dept_stats)
comparison_df


---
## 6. Actionable Decision Layer: จัดคิวสั่งการ กทม. (4 Quadrants Matrix)
แปลงระยะเวลาที่ทำนายได้ + ระดับความรุนแรง (Severity) + ความหนาแน่นประชากร (Density Factor) เป็น **Public Impact Score** และจัดกลุ่ม Triage สั่งการ


In [ ]:
from src.decision_layer import BMADecisionLayer

decision_layer = BMADecisionLayer()

# ประเมิน Triage กับตัวอย่าง 1,000 เคส
sample_test = test_eval_df.head(1000).copy()
triage_results = decision_layer.evaluate_dataframe(sample_test, duration_col='specialized_pred_days')

# สรุปสัดส่วนแต่ละ Quadrant
print("สรุปสัดส่วนการกระจายตัวของเคสใน 4 Quadrants:")
print(triage_results['triage_quadrant'].value_counts())

# แสดงตัวอย่างเคสเร่งด่วน Critical Urgent
print("
ตัวอย่างเคสวิกฤตเร่งด่วน (Critical Urgent - ต้องจบใน 24 ชม.):")
critical_cases = triage_results[triage_results['triage_quadrant'] == 'Critical Urgent']
critical_cases[['district', 'main_type', 'sub_category', 'severity', 'specialized_pred_days', 'public_impact_score', 'bma_action']].head(3)


---
## 7. Interactive Prediction Demo: ทดสอบจำลองรับเรื่องจริง
คุณสามารถทดลองเปลี่ยนข้อความและเขตด้านล่าง เพื่อดูว่าระบบ Route งานเข้าฝ่ายไหน และคาดการณ์เวลาซ่อมกี่วัน


In [ ]:
def test_new_complaint(district, main_type, sub_category, predicted_dept, severity, comment):
    case = {
        'district': district,
        'main_type': main_type,
        'sub_category': sub_category,
        'severity': severity,
        'comment_len': len(comment),
        'day_of_week': 2,        # วันพุธ
        'is_weekend': 0,
        'month': 6,              # เดือนมิถุนายน
        'is_rainy_season': 1,    # หน้าฝน
        'hour': 9                # 09:00 น.
    }
    
    # 1. ทำนายเวลาซ่อมด้วย Specialized Sub-models
    pred_res = predictor.predict_single(case, predicted_dept=predicted_dept)
    pred_days = pred_res['predicted_days']
    
    # 2. จัดคิว Actionable Triage
    triage = decision_layer.triage_ticket(district, severity, pred_days)
    
    print("=" * 60)
    print("📋 ผลการวิเคราะห์เรื่องร้องเรียน Traffy Fondue")
    print("=" * 60)
    print(f"พื้นที่แจ้งเหตุ : เขต{district}")
    print(f"ข้อความแจ้งเหตุ : '{comment}'")
    print(f"ฝ่ายรับผิดชอบ  : {predicted_dept}")
    print(f"โมเดลที่เลือกใช้ : {pred_res['model_used']} (Slug: {pred_res['model_slug']})")
    print("-" * 60)
    print(f"⏱️ เวลาคาดการณ์  : {pred_days} วัน (~{pred_days*24:.1f} ชั่วโมง)")
    print(f"🚨 Triage กลุ่ม : {triage['quadrant']}")
    print(f"💥 Impact Score: {triage['impact_score']}")
    print(f"📢 คำสั่งการ กทม.: {triage['action']}")
    print("=" * 60)

# ทดสอบรันเคสตัวอย่าง
test_new_complaint(
    district="จตุจักร",
    main_type="ขยะ",
    sub_category="ขยะตกค้างส่งกลิ่นเหม็น",
    predicted_dept="ฝ่ายรักษาความสะอาดฯ",
    severity=3,
    comment="ถังขยะหน้าปากซอยล้นมา 2 วันแล้ว ส่งกลิ่นเหม็นรบกวนคนเดินผ่านไปมา"
)
